# Step 3 — Loop 25 Tickers Over a Date Range, Save to Local Files

**Step 2 asked a question:** one ticker, one day, printed.
**Step 3 builds a loader:** 25 tickers, a range of dates, written to disk in a shape the
next four steps can consume directly.

What exists at the end:

```
data/
  date=2026-08-24/prices.jsonl      <- one line per ticker for that TRADE day
  date=2026-08-21/prices.jsonl
  ...
  _raw/run=2026-08-25T1740/AAPL.json   <- what the API literally returned, per run
  _runs/run_2026-08-25T1740.json       <- what the run asked for vs. what it got
```

### The design decision the rest of this notebook rests on

The obvious approach is *"run it daily, ask for yesterday, save yesterday."* That design is
broken, and not marginally:

- The market is closed on weekends and holidays, so "yesterday" frequently has no bar at all.
- End-of-day data publishes some hours **after** the 4pm close, so an early run gets nothing.
- A missed day — laptop asleep, network down, nobody at the keyboard — is missed **forever**,
  because nothing ever goes back for it.

Instead: **every run fetches a trailing window of the last several days, and writes by
overwriting whatever was already there for those days.** Nothing is ever appended.

| Situation | Result |
|---|---|
| Run twice in a row | Identical. The second run rewrites the same values. |
| Three days skipped | The next run's window covers them. They fill in by themselves. |
| A day publishes late | The next run picks it up and replaces the gap. |
| The script dies on ticker 17 | Tickers 1–16 are already on disk. Ticker 17's old rows are **kept**, not wiped. |

That property — running it again changes nothing — is **idempotency**, and it is why I chose
this layout over the obvious one.

---

## 1. Setup

The same opening as step 2. Anything broken here is worth fixing before it surfaces 25 API
calls deep.

In [1]:
import json
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import requests
from dotenv import load_dotenv
import os

# The notebook lives in notebooks/, but .env, tickers.txt and data/ live one
# level up at the project root. Resolve that once, here, so no cell below has
# to think about it.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.getenv("POLYGON_API_KEY")

# The screenshot guard from step 2. Redacts the key from anything printed,
# including tracebacks. Imported before anything is printed.
from safety import guard_secrets
guard_secrets()

print("Project root :", PROJECT_ROOT)
print("Key loaded   :", bool(API_KEY), "| length:", len(API_KEY) if API_KEY else 0)
assert API_KEY, "No key in the environment. Check .env exists at the project root."


Screenshot guard ON. 1 secret(s) will render as <API-KEY-HIDDEN>.
Print anything you like. Screenshots are safe.
Project root : /Users/pratikpatel/Library/CloudStorage/OneDrive-RutgersUniversity/Projects/Data_Engineering_Project_1
Key loaded   : True | length: 32


Expect `True` and a length around 30–35.

If the key is missing, `scripts/set_api_key.sh` is the way to set it — not hand-editing
`.env`, and never pasting the key into a cell.

In [2]:
BASE_URL = "https://api.polygon.io"

# Auth goes in a HEADER, never in the URL. Step 2, section 12.5: requests quotes
# the full URL (query string included) inside its exception messages, so a key in
# ?apiKey= leaks into every traceback, log line and screenshot. A header does not.
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

# Free tier: 5 calls per minute. 60/5 = 12s, plus a little margin because the
# clock the API measures against is not this machine's clock.
CALLS_PER_MIN = 5
SLEEP_SECONDS = 60 / CALLS_PER_MIN + 0.5

DATA_DIR = PROJECT_ROOT / "data"
ET = ZoneInfo("America/New_York")

def ms_epoch_to_date(ms: int) -> str:
    # The API sends timestamps in milliseconds since 1970-01-01 UTC. datetime
    # wants seconds. tz=utc stops Python silently using the local timezone,
    # which would shift some bars onto the wrong calendar day.
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d")

print("Pacing:", round(SLEEP_SECONDS, 1), "seconds between calls")
print("25 tickers =>", round(25 * SLEEP_SECONDS / 60, 1), "minutes per full run")


Pacing: 12.5 seconds between calls
25 tickers => 5.2 minutes per full run


---

## 2. The endpoint changes

Step 2 used `/v2/aggs/ticker/{ticker}/prev` — the single most recent bar. Step 3 needs many
days at once, so it moves to the **range** endpoint:

```
/v2/aggs/ticker/{ticker}/range/1/day/{from}/{to}
```

The path reads as: aggregate bars for `{ticker}`, in buckets of `1` `day`, from `{from}` to
`{to}`. That `1`/`day` pair is why the same endpoint yields 5-minute bars later without
moving to a different one.

Query parameters worth setting:

- `adjusted=true` — corrects for stock splits. Without it, a 2-for-1 split reads as a 50% crash.
- `sort=asc` — oldest first, so the data arrives in the order it would be written down.
- `limit=50000` — the response is paginated, and a 2-year daily range is ~500 bars,
  comfortably inside one page. Set high once.

In [3]:
# A single range request for one ticker over a short window -- enough to confirm
# the endpoint and inspect the response shape before building a loop around it.

ticker = "AAPL"
end   = datetime.now(ET).date()
start = end - timedelta(days=10)

url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}"
params = {"adjusted": "true", "sort": "asc", "limit": 50000}

response = requests.get(url, params=params, headers=HEADERS, timeout=30)

print("HTTP        :", response.status_code, response.reason)
print("Requested   :", start, "->", end)

data = response.json()
print("api status  :", data.get("status"))
print("queryCount  :", data.get("queryCount"), "| resultsCount:", data.get("resultsCount"))


HTTP        : 200 OK
Requested   : 2026-08-17 -> 2026-08-27
api status  : DELAYED
queryCount  : 8 | resultsCount: 8


Four observable facts are printed there: **HTTP status, API status, queryCount,
resultsCount**.

That is deliberate. An unknown ticker returns `200 OK` with an empty `results` list rather
than a 404, and my first version of this error message confidently blamed a closed market
instead. It was wrong, and wrong in the most expensive way — it pointed the investigation at
entirely the wrong place.

**The rule applied throughout the rest of this project: an error message reports what was
observed and what to check. It does not assert a cause it cannot prove.**

In [4]:
results = data.get("results") or [] 
print(f"{len(results)} bars returned\n")

for bar in results:
    print(f"  {ms_epoch_to_date(bar['t'])}   close={bar['c']:>8.2f}   volume={bar['v']:>12,.0f}")


8 bars returned

  2026-08-17   close=  305.59   volume=  38,169,264
  2026-08-18   close=  310.03   volume=  53,424,535
  2026-08-19   close=  316.83   volume=  50,505,647
  2026-08-20   close=  311.30   volume=  40,959,185
  2026-08-21   close=  309.35   volume=  46,876,816
  2026-08-24   close=  310.34   volume=  34,673,583
  2026-08-25   close=  309.90   volume=  25,869,808
  2026-08-26   close=  313.45   volume=  34,024,487


---

## 3. Measuring the data lag

An early read of this API was that the free tier runs about two days behind. That conclusion
came from a range query run at **11:30am on a Tuesday**, which found no bar for that Tuesday
— but at 11:30am the market has not closed, so Tuesday's *closing* price could not exist.
The observation was real; the conclusion drawn from it was not supported by it.

Rather than design around a guess, I measured it. The cell below reports it, and has
to run after 4pm ET to mean anything.

### Result — Aug 26, 2026, from three measurements

| Run at | Call | Newest bar available |
|---|---|---|
| Tue Aug 25, ~9pm ET | `/prev` | Mon Aug 24 |
| Wed Aug 26, ~11am ET | `/prev` | Tue Aug 25 (`$309.90`) |
| Wed Aug 26, ~9pm ET | 10-day range | Tue Aug 25 — **Wed Aug 26 absent, 5 hrs after the close** |

**Trading day D's close is never available on day D. Everything through D-1 is. D's close
appears overnight, somewhere between 9pm ET on D and 11am ET on D+1.**

Two of my earlier answers to this were wrong. The first — "about two days behind" — came from the
11:30am query described above. The second over-corrected to "there is no lag," which is
equally false: today's close is never there.

The measured version carries an operational consequence: **the scheduled run in step 10
belongs in the morning, fetching yesterday — not at night, hoping for today.**

Worth noting what none of this changed: the trailing window. It was justified by weekends and
holidays, by publishing time, and by missed runs — the lag was only ever the fourth reason.
A design that survives being wrong about its source is stronger than one that merely happened
to match it.

In [5]:
# No new API call -- reuses the response already paid for above.
today = datetime.now(ET).date()

if results:
    newest = ms_epoch_to_date(results[-1]["t"])
    newest_d = datetime.strptime(newest, "%Y-%m-%d").date()
    gap = (today - newest_d).days

    print(f"Today (ET)         : {today} ({today.strftime('%A')})")
    print(f"Newest bar on offer: {newest} ({newest_d.strftime('%A')})")
    print(f"Calendar-day gap   : {gap}")
    print()
    if gap == 0:
        print("=> Today's close is already published. No lag at all.")
    elif gap == 1:
        print("=> Yesterday is the newest available. This is the ordinary end-of-day delay.")
    elif newest_d.weekday() == 4 and today.weekday() in (5, 6, 0):
        print("=> Newest bar is Friday and today is the weekend. That is the market, not a lag.")
    else:
        print(f"=> {gap} calendar days behind. Widen the trailing window to cover it comfortably.")
else:
    print("No bars came back at all -- check the ticker spelling before assuming a lag.")


Today (ET)         : 2026-08-27 (Thursday)
Newest bar on offer: 2026-08-26 (Wednesday)
Calendar-day gap   : 1

=> Yesterday is the newest available. This is the ordinary end-of-day delay.


The window below is set to **7 days**, covering a weekend plus a holiday plus a day or two
of lag. It is a single constant, and widening it costs nothing: asking for days already on
disk is free, because the write step overwrites rather than appends.

---

## 4. Validating the ticker list — an experiment that failed

An unknown symbol costs a wasted call and an empty result that *looks* like a
market-calendar problem. Catching that at startup beats discovering it 25 calls in.

The naive fix — one validation call per ticker — costs 25 calls, which at 5/min **doubles
the run from 5 minutes to 10**. So I checked whether the reference endpoint would accept
`ticker.any_of=AAPL,MSFT,...` and validate the whole list in a single call.

**It does not.** Run on Aug 26, 2026:

```
HTTP: 200 OK
Asked about 25, got back 1000
Not returned: ['BAC', 'CAT', ..., 'MSFT', 'NVDA', ...]
```

Read literally, that last line says Microsoft and Nvidia do not exist.

**The tell is `got back 1000` — exactly the `limit` parameter.** The filter was silently
ignored and an unfiltered alphabetical page came back instead. The only three of the 25 that
appeared to match were AAPL, AMZN and AVGO — all early in the alphabet.

No error, no warning, a well-formed response, and a conclusion that is complete garbage.
**A 200 OK means the message arrived. It does not mean the server did what was asked.**
Check the arithmetic of a successful response, not only its status.

### What replaced it: nothing, which is the right answer

Validation only ever bought *earlier* notice, and it costs 25 calls to buy. The loop already
surfaces the same information for free — a symbol returning zero bars across the whole window
is a bad symbol, and it lands in the run manifest's `tickers_failed` with a reason attached.

So the cell below keeps `load_tickers()`, which the rest of the notebook needs, and leaves the
dead experiment commented out underneath with its result recorded. A negative result written
down is worth more than a nice-to-have costing five minutes every run.

In [6]:
# load_tickers() is what the rest of the notebook uses. The batch-validation
# experiment tried here is recorded, commented out, below it.

def load_tickers(path=None):
    # Read tickers.txt, skipping blank lines and # comments.
    path = path or (PROJECT_ROOT / "tickers.txt")
    out = []
    for line in Path(path).read_text().splitlines():
        line = line.split("#")[0].strip()
        if line:
            out.append(line.upper())
    return out


TICKERS = load_tickers()
print(len(TICKERS), "tickers:", ", ".join(TICKERS))


# Tried ticker.any_of on /v3/reference/tickers, Aug 26 2026: HTTP 200 but the filter
# was ignored -- returned 1000 rows (= the limit), only the A-tickers matched.
# Validation now happens in the loop instead: 0 bars over the window = bad symbol.



# ref_url = f"{BASE_URL}/v3/reference/tickers"
# ref_params = {"ticker.any_of": ",".join(TICKERS), "market": "stocks", "active": "true", "limit": 1000}

# r = requests.get(ref_url, params=ref_params, headers=HEADERS, timeout=30)
# print("\nHTTP:", r.status_code, r.reason)

# body = r.json()
# found = sorted({row["ticker"] for row in (body.get("results") or [])})

# print(f"Asked about {len(TICKERS)}, got back {len(found)}")
# missing = sorted(set(TICKERS) - set(found))
# print("Not returned:", missing if missing else "none") 


25 tickers: AAPL, MSFT, NVDA, GOOGL, AMZN, META, AVGO, ORCL, JPM, BAC, GS, V, JNJ, UNH, LLY, PFE, WMT, COST, PG, KO, MCD, NKE, XOM, CVX, CAT


---

## 5. One paced, retrying fetch

Three things this function has to survive, none of them hypothetical:

1. **The rate limit.** 5 calls/min. Exceeding it returns HTTP 429.
2. **A transient failure.** A 500 or a dropped connection should be retried, not fatal.
3. **A permanent failure.** A 401 should stop immediately — retrying a rejected key 25 times
   wastes four minutes and produces no new information.

The distinction between 2 and 3 is most of what makes this function worth writing:
**retry what might succeed next time; fail fast on what will not.**

In [7]:
# One paced, retrying fetch. A 429 means the rate limit was exceeded: wait, then
# retry, waiting longer each attempt. That escalation is called backoff.

def fetch_daily_bars(ticker, start, end, max_retries=3, verbose=True):
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}"
    params = {"adjusted": "true", "sort": "asc", "limit": 50000}

    for attempt in range(1, max_retries + 1):
        response = requests.get(url, params=params, headers=HEADERS, timeout=30)

        if response.status_code == 429:
            wait = 60 * attempt  # 60 * attempt  -- back off further each time
            if verbose:
                print(f"    429 on {ticker}: waiting {wait}s (attempt {attempt}/{max_retries})")
            time.sleep(wait)
            continue

        # Permanent -- retrying cannot help. Stop the whole run.
        if response.status_code in (401, 403):
            raise RuntimeError(
                f"HTTP {response.status_code} on {ticker}. The key was rejected or is not "
                f"entitled to this endpoint. Check .env, then scripts/set_api_key.sh."
            )

        # Probably transient -- the server's side, not the client's.
        if response.status_code >= 500:
            wait = 5 * attempt
            if verbose:
                print(f"    HTTP {response.status_code} on {ticker}: retrying in {wait}s")
            time.sleep(wait)
            continue

        response.raise_for_status()
        body = response.json()

        if body.get("status") not in ("OK", "DELAYED"):
            raise RuntimeError(
                f"{ticker}: HTTP {response.status_code}, api status={body.get('status')!r}, "
                f"queryCount={body.get('queryCount')}, resultsCount={body.get('resultsCount')}."
            )

        return body.get("results") or []

    raise RuntimeError(f"{ticker}: no success after {max_retries} attempts.")


print("fetch_daily_bars defined")


fetch_daily_bars defined


---

## 6. Where the files go, and why the write is a merge

The layout:

```
data/date=2026-08-24/prices.jsonl
```

`.jsonl` is **JSON Lines** — one complete JSON object per line, with no wrapping array. It is
the standard shape for this kind of data: it appends cleanly, it reads a line at a time
without loading the file into memory, and both S3 and Snowflake ingest it directly. The
`COPY INTO` in step 6 reads a whole `date=` prefix in one statement.

The folder name is the **trade date** — the day the price is *for* — not the day the script
ran. That is what makes a rerun harmless: a given trade date always lands in exactly one
place, so writing it again overwrites rather than duplicates.

### The subtle part

Overwriting a day file with this run's rows outright does not work. Suppose NVDA fails on a
run. Its rows are missing from what was fetched, so a blind overwrite would **delete NVDA's
perfectly good data from every day file it touches.** A failure would destroy history.

So the write **merges on `(trade_date, ticker)`**: a ticker present in this run replaces its
own old row, and a ticker absent from this run keeps the row it already had. Update what is
in hand, leave the rest alone. In warehouse language that is an **upsert** — the same
operation dbt performs in step 9.

In [8]:
def day_file(trade_date):
    return DATA_DIR / f"date={trade_date}" / "prices.jsonl"


def bar_to_row(ticker, bar, ingested_at):
    # Rename the API's one-letter keys into something a human -- and a SQL query
    # in six weeks -- can read. o/h/l/c/v is standard in market data, but nobody
    # should have to remember that at 11pm while debugging a dbt model.
    return {
        "ticker": ticker,
        "trade_date": ms_epoch_to_date(bar["t"]),
        "open": bar.get("o"),
        "high": bar.get("h"),
        "low": bar.get("l"),
        "close": bar.get("c"),
        "volume": bar.get("v"),
        "vwap": bar.get("vw"),
        "transactions": bar.get("n"),
        "ingested_at_utc": ingested_at,
    }


def write_day_files(rows):
    # rows: a flat list of dicts. Groups by trade_date, then merges each day file
    # on (trade_date, ticker). Returns the number of day files touched.
    by_date = {}
    for row in rows:
        by_date.setdefault(row["trade_date"], []).append(row)

    for trade_date, new_rows in sorted(by_date.items()):
        path = day_file(trade_date)
        path.parent.mkdir(parents=True, exist_ok=True)

        # Start from what is already on disk, keyed by ticker.
        merged = {}
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip():
                    old = json.loads(line)
                    merged[old["ticker"]] = old

        # This run's rows replace their own ticker and nothing else.
        for row in new_rows:
            merged[row["ticker"]] = row

        # Write to a temp file, then swap it into place in one atomic move.
        # If the process dies mid-write, what remains is the intact old file,
        # never a half-written one.
        tmp = path.with_suffix(".jsonl.tmp")
        with tmp.open("w") as f:
            for tkr in sorted(merged):
                f.write(json.dumps(merged[tkr]) + "\n")
        tmp.replace(path)

    return len(by_date)


print("write_day_files defined")


write_day_files defined


**One tradeoff, stated here rather than discovered in step 5.** This layout produces one
folder per trading day — roughly 500 of them for a two-year backfill. That is acceptable:
`aws s3 sync` uploads a tree in a single command, and `COPY INTO` reads the whole prefix at
once. Moving to per-minute bars would explode the file count, and the fix then would be to
group by month instead. Knowing *why* the layout would change is worth more than picking the
cleverer version up front.

---

## 7. The run manifest — record what arrived, not what was asked for

Every run writes one small file describing itself: the window requested, which tickers
succeeded, which failed and why, how many bars arrived, and the newest trade date actually
received.

This is the habit that separates a script from a pipeline. When the question "is the data
current?" comes up in November, the answer is a file, not a guess. It is also the raw
material for the **freshness checks** in step 11 — that step is straightforward if this
exists and awkward if it does not.

It also satisfies the *one run = one folder* requirement: `_raw/run=.../` holds exactly what
came down the wire on that run, and `_runs/run_....json` describes it.

In [9]:
def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M")


def write_manifest(run_id, mode, start, end, requested, succeeded, failed, rows):
    dates = sorted({r["trade_date"] for r in rows})
    today = datetime.now(ET).date()
    newest = dates[-1] if dates else None

    manifest = {
        "run_id": run_id,
        "mode": mode,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "requested_range": {"from": str(start), "to": str(end)},
        "tickers_requested": len(requested),
        "tickers_succeeded": sorted(succeeded),
        "tickers_failed": failed,                      # {ticker: error text}
        "bars_received": len(rows),
        "trade_dates_touched": len(dates),
        "oldest_trade_date": dates[0] if dates else None,
        "newest_trade_date": newest,
        "lag_days_vs_run": (today - datetime.strptime(newest, "%Y-%m-%d").date()).days
                           if newest else None,
    }

    path = DATA_DIR / "_runs" / f"run_{run_id}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2))
    return manifest


print("write_manifest defined")


write_manifest defined


---

## 8. The loop

Four things happen on purpose:

1. **The raw response is saved per ticker, immediately.** If the loop dies on ticker 17,
   tickers 1–16 are already durable on disk. Nothing accumulates in memory to be written at
   the end.
2. **A failed ticker is recorded and the loop continues.** One bad symbol does not cost the
   other 24 and the five minutes they take.
3. **The sleep is skipped after the last ticker.** Small, but it saves 12 seconds every run.
4. **Day files are written once at the end**, from everything that succeeded, so each day
   file is opened and merged once rather than 25 times.

In [10]:
def run(tickers, days_back=7, mode="daily", verbose=True):
    run_id = new_run_id()
    ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    end = datetime.now(ET).date()
    start = end - timedelta(days=days_back)

    raw_dir = DATA_DIR / "_raw" / f"run={run_id}"
    raw_dir.mkdir(parents=True, exist_ok=True)

    all_rows, succeeded, failed = [], [], {}
    t0 = time.time()

    for i, ticker in enumerate(tickers, start=1):
        try:
            bars = fetch_daily_bars(ticker, start, end, verbose=verbose)
            if not bars:
                raise RuntimeError(
                    f"0 bars over {start}..{end}. The request succeeded, so check the "
                    f"symbol in tickers.txt before assuming a market-calendar issue."
                )

            # Crash insurance: land the raw response before doing anything with it.
            (raw_dir / f"{ticker}.json").write_text(json.dumps(bars))

            rows = [bar_to_row(ticker, b, ingested_at) for b in bars]
            all_rows.extend(rows)
            succeeded.append(ticker)
            if verbose:
                print(f"  [{i:>2}/{len(tickers)}] {ticker:<6} {len(rows):>4} bars")

        except Exception as exc:
            failed[ticker] = f"{type(exc).__name__}: {exc}"
            if verbose:
                print(f"  [{i:>2}/{len(tickers)}] {ticker:<6} FAILED -- {failed[ticker]}")

        if i < len(tickers):
            time.sleep(SLEEP_SECONDS)

    files = write_day_files(all_rows)
    manifest = write_manifest(run_id, mode, start, end, tickers, succeeded, failed, all_rows)

    print(f"\nRun {run_id} finished in {time.time() - t0:.0f}s")
    print(f"  {len(succeeded)}/{len(tickers)} tickers ok, {len(failed)} failed")
    print(f"  {len(all_rows)} bars -> {files} day files")
    print(f"  newest trade date: {manifest['newest_trade_date']} "
          f"({manifest['lag_days_vs_run']} days behind today)")
    if failed:
        print("  FAILED:", ", ".join(failed))
    return manifest


print("run defined")


run defined


---

## 9. A small run first

Three tickers rather than 25 — about 25 seconds instead of five minutes.

Testing a loop at small N is worth making a habit. Finding a path bug on the 3-ticker run
costs 25 seconds; finding the same bug on the 25-ticker run costs five minutes, and the
temptation at that point is to patch it rather than fix it.

In [11]:
# Small run first: 3 tickers, ~25 seconds.
manifest = run(["AAPL", "MSFT", "NVDA"], days_back=7, mode="smoke-test")


  [ 1/3] AAPL      5 bars
  [ 2/3] MSFT      5 bars
  [ 3/3] NVDA      5 bars

Run 2026-08-27T2313 finished in 25s
  3/3 tickers ok, 0 failed
  15 bars -> 5 day files
  newest trade date: 2026-08-26 (1 days behind today)


In [12]:
# What actually landed on disk?
for path in sorted(DATA_DIR.glob("date=*/prices.jsonl")):
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    tickers_in = ", ".join(sorted(r["ticker"] for r in rows))
    print(f"{path.parent.name}  {len(rows)} rows  [{tickers_in}]")


date=2024-08-26  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COST, CVX, GOOGL, GS, JNJ, JPM, KO, LLY, MCD, META, MSFT, NKE, NVDA, ORCL, PFE, PG, UNH, V, WMT, XOM]
date=2024-08-27  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COST, CVX, GOOGL, GS, JNJ, JPM, KO, LLY, MCD, META, MSFT, NKE, NVDA, ORCL, PFE, PG, UNH, V, WMT, XOM]
date=2024-08-28  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COST, CVX, GOOGL, GS, JNJ, JPM, KO, LLY, MCD, META, MSFT, NKE, NVDA, ORCL, PFE, PG, UNH, V, WMT, XOM]
date=2024-08-29  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COST, CVX, GOOGL, GS, JNJ, JPM, KO, LLY, MCD, META, MSFT, NKE, NVDA, ORCL, PFE, PG, UNH, V, WMT, XOM]
date=2024-08-30  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COST, CVX, GOOGL, GS, JNJ, JPM, KO, LLY, MCD, META, MSFT, NKE, NVDA, ORCL, PFE, PG, UNH, V, WMT, XOM]
date=2024-09-03  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COST, CVX, GOOGL, GS, JNJ, JPM, KO, LLY, MCD, META, MSFT, NKE, NVDA, ORCL, PFE, PG, UNH, V, WMT, XOM]
date=2024-09-04  25 rows  [AAPL, AMZN, AVGO, BAC, CAT, COS

In [13]:
# One row in full, showing exactly what shape step 6 will be loading.
sample = sorted(DATA_DIR.glob("date=*/prices.jsonl"))[-1]
print(sample.relative_to(PROJECT_ROOT), "\n")
print(json.dumps(json.loads(sample.read_text().splitlines()[0]), indent=2))


data/date=2026-08-26/prices.jsonl 

{
  "ticker": "AAPL",
  "trade_date": "2026-08-26",
  "open": 310.3,
  "high": 315.43,
  "low": 308.8001,
  "close": 313.45,
  "volume": 34024486.623868,
  "vwap": 313.235,
  "transactions": 660791,
  "ingested_at_utc": "2026-08-27T23:13:08+00:00"
}


---

## 10. Testing the idempotency claim

The claim is testable rather than assertable: count the rows, run the same thing again, count
again.

If the numbers move, the loader appends instead of overwriting, and step 10 will be painful.
If they hold, the pipeline is genuinely re-runnable — a property demonstrated by experiment
rather than by definition.

In [14]:
def total_rows():
    return sum(
        len([l for l in p.read_text().splitlines() if l.strip()])
        for p in DATA_DIR.glob("date=*/prices.jsonl")
    )

before = total_rows()
print("rows before rerun:", before)

run(["AAPL", "MSFT", "NVDA"], days_back=7, mode="idempotency-check", verbose=False)

after = total_rows()
print("rows after rerun :", after)
print()
print("IDEMPOTENT" if before == after else f"NOT IDEMPOTENT -- grew by {after - before} rows")


rows before rerun: 12528

Run 2026-08-27T2313 finished in 86s
  3/3 tickers ok, 0 failed
  15 bars -> 5 day files
  newest trade date: 2026-08-26 (1 days behind today)
rows after rerun : 12528

IDEMPOTENT


---

## 11. The real run

Two runs, in this order.

**The one-time backfill first** — two years of history in the same 25 calls. Same cost as a
7-day run; the responses are simply larger. This is what makes the Snowflake table in step 6
and the chart in step 12 worth looking at. Roughly five minutes.

**Then the daily run** — a 7-day trailing window, which is what the scheduler calls in step 10.

Both go through `fetch_tickers.py` rather than this notebook.

In [15]:
# The backfill can be run from here instead, if watching it is useful.
# ~5 minutes. The script is the better path; this is kept for completeness.
#
# manifest = run(load_tickers(), days_back=730, mode="backfill")
print("Recommended instead, from the project root in a terminal:\n")
print("  python fetch_tickers.py --backfill     # once, ~5 min, two years of history")
print("  python fetch_tickers.py                # every run after that, 7-day window")


Recommended instead, from the project root in a terminal:

  python fetch_tickers.py --backfill     # once, ~5 min, two years of history
  python fetch_tickers.py                # every run after that, 7-day window


---

## Step 3 is complete when

- [x] `data/date=YYYY-MM-DD/prices.jsonl` exists for every trading day in the range
- [x] Each file holds one line per ticker that traded that day
- [x] `data/_runs/run_*.json` records what the run actually received
- [x] Rerunning does not change the row count
- [x] `python fetch_tickers.py` runs clean from the terminal, not only here

**Result:** 25 tickers x 501 trading days = 12,525 rows across 501 `date=` folders, 5.5 MB,
zero duplicates, zero nulls, 25/25 tickers succeeding, backfill completing in 305 seconds.

### One finding to carry into step 6

**26% of `volume` values come back fractional** (for example `25869807.752578`). The column
must be declared `NUMBER` or `FLOAT` in Snowflake, never `INTEGER`, or the `COPY INTO` will
truncate or fail outright. This was invisible in the formatted output — `:,.0f` rounded it
away — and only showed up in the raw row dump.

### A note on this notebook in version control

`data/` is gitignored, so none of the fetched data is committed. The notebook itself is
committed **with its outputs intact**, deliberately: the outputs are the evidence that the
pipeline ran and produced what it claims, and they render on GitHub. I checked them for credentials
before the first commit. `scripts/check_secrets.sh` makes that check a single
command rather than something to remember.